# Transcoders: Interpretable MLP Feature Circuits

Transcoders (Dunefsky et al., NeurIPS 2024) replace dense MLP sublayers with
sparse, overcomplete alternatives. Unlike SAEs (which reconstruct residual
stream activations), transcoders learn the input-to-output mapping of MLPs,
enabling weight-based circuit analysis through MLP layers.

This notebook covers the `irtk.transcoder` module.

## Why This Matters

Transcoders learn to map between MLP input and output activations, providing a sparse decomposition of what each MLP layer computes. Unlike SAEs (which decompose a single activation), transcoders capture the input-output transformation, revealing MLP features as computational primitives.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import transcoder

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Train Transcoders

Train transcoders for two MLP layers to enable circuit analysis.

In [ ]:
prompts = [
    "The Eiffel Tower is located in",
    "The capital of France is",
    "London is the capital of",
    "Berlin is a city in",
    "Tokyo is the capital of",
    "The weather today is very",
    "My favorite food is definitely",
    "The president of the United States",
]
token_seqs = [model.to_tokens(p) for p in prompts]

# Train transcoders for layers 6 and 8
tc6_result = transcoder.train_transcoder(
    model, layer=6, token_sequences=token_seqs,
    n_features=512, l1_coeff=5e-4, epochs=20, lr=3e-4
)
tc8_result = transcoder.train_transcoder(
    model, layer=8, token_sequences=token_seqs,
    n_features=512, l1_coeff=5e-4, epochs=20, lr=3e-4
)
tc6, tc8 = tc6_result.transcoder, tc8_result.transcoder

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tc6_result.train_losses, label='Train')
axes[0].plot(tc6_result.val_losses, label='Val')
axes[0].set_title('Layer 6 Transcoder Loss')
axes[0].legend()
axes[1].plot(tc8_result.train_losses, label='Train')
axes[1].plot(tc8_result.val_losses, label='Val')
axes[1].set_title('Layer 8 Transcoder Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. Feature-to-Feature Circuit

Build a weight-based connection matrix between features at different layers.

In [ ]:
result = transcoder.transcoder_feature_circuit(tc6, tc8)

fig, ax = plt.subplots(figsize=(10, 8))
# Show a subset of the connection matrix
n_show = 50
im = ax.imshow(result['connection_matrix'][:n_show, :n_show],
               cmap='RdBu_r', aspect='auto')
ax.set_xlabel('Layer 8 Features')
ax.set_ylabel('Layer 6 Features')
ax.set_title(f'Feature-to-Feature Connections (first {n_show})')
plt.colorbar(im, ax=ax, label='Connection strength')
plt.tight_layout()
plt.show()

print(f"Mean |connection|: {result['mean_connection']:.6f}")
print(f"\nTop 5 connections:")
for feat_a, feat_b, strength in result['top_connections'][:5]:
    print(f"  L6.F{feat_a} -> L8.F{feat_b}: {strength:+.4f}")

## 3. Top Activating Contexts

Find token contexts that maximally activate a transcoder feature.

In [ ]:
# Find most active feature
all_acts = []
for tokens in token_seqs:
    _, cache = model.run_with_cache(tokens)
    hook = 'blocks.6.hook_resid_mid'
    if hook not in cache.cache_dict:
        hook = 'blocks.6.hook_resid_pre'
    acts = cache.cache_dict[hook]
    all_acts.append(np.array(tc6.encode(acts)))
mean_acts = np.mean(np.concatenate(all_acts, axis=0), axis=0)
top_feat = int(np.argmax(mean_acts))

examples = transcoder.top_activating_for_transcoder_feature(
    tc6, model, 6, top_feat, token_seqs, k=5
)

print(f"Feature {top_feat} top activating contexts:")
for ex in examples:
    toks = [tokenizer.decode([int(t)]) for t in ex['tokens']]
    pos = ex['position']
    print(f"  act={ex['activation']:.3f}  token='{toks[pos]}'  context: {''.join(toks[:pos+1])}")

## 4. MLP Feature Logit Attribution

What output tokens does a transcoder feature promote or suppress?

In [ ]:
result = transcoder.mlp_feature_logit_attribution(tc6, model, top_feat, k=10)

print(f"Feature {top_feat} logit attribution:")
print("\nTokens promoted:")
for tid, effect in result['top_promoted'][:5]:
    print(f"  {tokenizer.decode([tid]):>15s}: {effect:+.4f}")

print("\nTokens suppressed:")
for tid, effect in result['top_suppressed'][:5]:
    print(f"  {tokenizer.decode([tid]):>15s}: {effect:+.4f}")

## Summary

| Function/Class | Purpose |
|----------|--------|
| `Transcoder` | Sparse MLP replacement (encode/decode via wide hidden layer) |
| `train_transcoder()` | Train a transcoder to approximate an MLP layer |
| `transcoder_feature_circuit()` | Weight-based feature-to-feature connections |
| `top_activating_for_transcoder_feature()` | Top-k activating contexts for a feature |
| `mlp_feature_logit_attribution()` | Feature's effect on output logits |